# 🏋️ Week 5: NLP & Deployment Practice

Practice NLP tasks and API design patterns.

---

In [ ]:
import numpy as np
import pandas as pd
import re
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
print("Imports ready!")

---
## Exercise 1: Text Preprocessing Pipeline

Build a complete text preprocessing function from scratch.

**Difficulty:** Medium | **Skill:** NLP fundamentals

In [ ]:
def preprocess_text(text: str, 
                    lowercase: bool = True,
                    remove_punctuation: bool = True,
                    remove_numbers: bool = True,
                    remove_stopwords: bool = True) -> str:
    """
    Preprocess text for NLP tasks.
    
    Steps:
    1. Lowercase
    2. Remove URLs and emails
    3. Remove punctuation
    4. Remove numbers
    5. Remove stopwords
    6. Remove extra whitespace
    
    Returns:
        Cleaned text string
    """
    # YOUR CODE HERE
    pass


# Test
sample = "Check out http://example.com for 50% OFF!!! Contact: test@email.com #Amazing #Deal"
result = preprocess_text(sample)
print(f"Original: {sample}")
print(f"Cleaned: {result}")

In [ ]:
# Solution
def preprocess_text_solution(text: str, 
                              lowercase: bool = True,
                              remove_punctuation: bool = True,
                              remove_numbers: bool = True,
                              remove_stopwords: bool = True) -> str:
    # Common stopwords
    STOPWORDS = {'the', 'a', 'an', 'is', 'are', 'was', 'were', 'be', 'been', 
                 'being', 'have', 'has', 'had', 'do', 'does', 'did', 'will',
                 'would', 'could', 'should', 'may', 'might', 'must', 'shall',
                 'can', 'to', 'of', 'in', 'for', 'on', 'with', 'at', 'by',
                 'from', 'as', 'into', 'through', 'during', 'before', 'after',
                 'above', 'below', 'up', 'down', 'out', 'off', 'over', 'under',
                 'and', 'but', 'or', 'nor', 'so', 'yet', 'both', 'either',
                 'neither', 'not', 'only', 'own', 'same', 'than', 'too', 'very',
                 'just', 'it', 'its', 'this', 'that', 'these', 'those', 'i', 'me',
                 'my', 'myself', 'we', 'our', 'ours', 'you', 'your', 'he', 'him',
                 'his', 'she', 'her', 'hers', 'they', 'them', 'their', 'what',
                 'which', 'who', 'whom', 'when', 'where', 'why', 'how', 'all',
                 'each', 'every', 'any', 'some', 'no', 'such'}
    
    if lowercase:
        text = text.lower()
    
    # Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)
    
    # Remove emails
    text = re.sub(r'\S+@\S+', '', text)
    
    if remove_punctuation:
        text = re.sub(r'[^\w\s]', '', text)
    
    if remove_numbers:
        text = re.sub(r'\d+', '', text)
    
    if remove_stopwords:
        words = text.split()
        words = [w for w in words if w not in STOPWORDS]
        text = ' '.join(words)
    
    # Remove extra whitespace
    text = ' '.join(text.split())
    
    return text

print(preprocess_text_solution(sample))

---
## Exercise 2: Build TF-IDF from Scratch

Implement TF-IDF vectorization without sklearn.

**Difficulty:** Hard | **Skill:** Understanding vectorization

In [ ]:
import math

def compute_tfidf(documents: list) -> tuple:
    """
    Compute TF-IDF from scratch.
    
    TF = (count of word in doc) / (total words in doc)
    IDF = log(total docs / docs containing word)
    TF-IDF = TF * IDF
    
    Args:
        documents: List of text documents
    
    Returns:
        Tuple of (tfidf_matrix, vocabulary)
    """
    # YOUR CODE HERE
    pass


# Test
docs = [
    "I love machine learning",
    "Machine learning is great",
    "I love deep learning too"
]

tfidf_matrix, vocab = compute_tfidf(docs)
print("Vocabulary:", vocab)
print("TF-IDF Matrix shape:", tfidf_matrix.shape)

In [ ]:
# Solution
def compute_tfidf_solution(documents: list) -> tuple:
    # Tokenize all documents
    tokenized = [doc.lower().split() for doc in documents]
    
    # Build vocabulary
    vocab = sorted(set(word for doc in tokenized for word in doc))
    word_to_idx = {word: i for i, word in enumerate(vocab)}
    
    n_docs = len(documents)
    n_vocab = len(vocab)
    
    # Compute document frequencies (how many docs contain each word)
    doc_freq = Counter()
    for doc in tokenized:
        doc_freq.update(set(doc))
    
    # Compute IDF
    idf = {}
    for word in vocab:
        idf[word] = math.log(n_docs / doc_freq[word])
    
    # Compute TF-IDF matrix
    tfidf_matrix = np.zeros((n_docs, n_vocab))
    
    for i, doc in enumerate(tokenized):
        word_counts = Counter(doc)
        total_words = len(doc)
        
        for word, count in word_counts.items():
            tf = count / total_words
            j = word_to_idx[word]
            tfidf_matrix[i, j] = tf * idf[word]
    
    return tfidf_matrix, vocab

tfidf_matrix, vocab = compute_tfidf_solution(docs)
print("Vocabulary:", vocab)
print("\nTF-IDF Matrix:")
print(pd.DataFrame(tfidf_matrix, columns=vocab).round(3))

---
## Exercise 3: Sentiment Classifier

Build a simple sentiment classifier using TF-IDF and Logistic Regression.

**Difficulty:** Medium | **Skill:** Text classification

In [ ]:
# Sample sentiment data
sentiment_data = pd.DataFrame({
    'text': [
        "I love this product, it's amazing!",
        "Terrible experience, would not recommend",
        "Great quality and fast shipping",
        "Waste of money, very disappointed",
        "Excellent customer service!",
        "The worst purchase I ever made",
        "Highly recommend to everyone",
        "Never buying from them again",
        "Best product in the market",
        "Complete garbage, total scam"
    ],
    'sentiment': [1, 0, 1, 0, 1, 0, 1, 0, 1, 0]
})

def build_sentiment_classifier(df: pd.DataFrame, text_col: str, label_col: str):
    """
    Build a sentiment classifier pipeline.
    
    Steps:
    1. Vectorize text with TF-IDF
    2. Train Logistic Regression
    3. Return model and vectorizer
    
    Returns:
        Dict with model, vectorizer, accuracy
    """
    # YOUR CODE HERE
    pass


result = build_sentiment_classifier(sentiment_data, 'text', 'sentiment')
print(f"Accuracy: {result['accuracy']:.2%}")

In [ ]:
# Solution
def build_sentiment_classifier_solution(df: pd.DataFrame, text_col: str, label_col: str):
    # Vectorize
    vectorizer = TfidfVectorizer(max_features=100, stop_words='english')
    X = vectorizer.fit_transform(df[text_col])
    y = df[label_col]
    
    # Split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=42
    )
    
    # Train
    model = LogisticRegression(max_iter=1000)
    model.fit(X_train, y_train)
    
    # Evaluate
    accuracy = model.score(X_test, y_test)
    
    return {
        'model': model,
        'vectorizer': vectorizer,
        'accuracy': accuracy
    }

result = build_sentiment_classifier_solution(sentiment_data, 'text', 'sentiment')

# Test on new text
def predict_sentiment(text, model, vectorizer):
    vec = vectorizer.transform([text])
    pred = model.predict(vec)[0]
    return 'Positive' if pred == 1 else 'Negative'

test_texts = [
    "This is absolutely wonderful!",
    "Horrible, don't buy this"
]

for text in test_texts:
    pred = predict_sentiment(text, result['model'], result['vectorizer'])
    print(f"{text[:30]}... → {pred}")

---
## Exercise 4: API Request/Response Schema Design

Design proper API schemas for ML model serving.

**Difficulty:** Easy | **Skill:** API design

In [ ]:
from dataclasses import dataclass
from typing import List, Optional

@dataclass
class PredictionRequest:
    """Design a request schema for a sentiment prediction API."""
    # YOUR CODE HERE - Add appropriate fields
    pass


@dataclass
class PredictionResponse:
    """Design a response schema for a sentiment prediction API."""
    # YOUR CODE HERE - Add appropriate fields
    pass


# Example usage:
# request = PredictionRequest(text="Great product!")
# response = PredictionResponse(sentiment="positive", confidence=0.95)

In [ ]:
# Solution
from dataclasses import dataclass, asdict
from typing import List, Optional
import json

@dataclass
class PredictionRequestSolution:
    """Request schema for sentiment prediction."""
    text: str
    return_probabilities: bool = True
    threshold: float = 0.5
    

@dataclass
class PredictionResponseSolution:
    """Response schema for sentiment prediction."""
    sentiment: str
    confidence: float
    probabilities: Optional[dict] = None
    model_version: str = "1.0.0"
    

@dataclass
class BatchPredictionRequest:
    """Batch prediction request."""
    texts: List[str]
    return_probabilities: bool = True
    

@dataclass  
class BatchPredictionResponse:
    """Batch prediction response."""
    predictions: List[PredictionResponseSolution]
    total_processed: int
    processing_time_ms: float


# Demo
request = PredictionRequestSolution(text="Great product!")
response = PredictionResponseSolution(
    sentiment="positive",
    confidence=0.95,
    probabilities={"positive": 0.95, "negative": 0.05}
)

print("Request:")
print(json.dumps(asdict(request), indent=2))
print("\nResponse:")
print(json.dumps(asdict(response), indent=2))

---
## Exercise 5: Model Serving Function

Create a production-ready prediction function with error handling.

**Difficulty:** Medium | **Skill:** Production code

In [ ]:
def serve_prediction(text: str, model, vectorizer, threshold: float = 0.5) -> dict:
    """
    Production-ready prediction function.
    
    Features:
    - Input validation
    - Error handling
    - Confidence scores
    - Proper response format
    
    Returns:
        Dict with prediction, confidence, and status
    """
    # YOUR CODE HERE
    pass


# Test with various inputs
test_cases = [
    "This is amazing!",
    "",  # Empty string
    "A" * 10000,  # Very long string
    None  # Invalid input
]

for text in test_cases:
    result = serve_prediction(text, result['model'], result['vectorizer'])
    print(f"Input: {str(text)[:30]}... → {result}")

In [ ]:
# Solution
def serve_prediction_solution(text: str, model, vectorizer, threshold: float = 0.5) -> dict:
    # Input validation
    if text is None:
        return {'status': 'error', 'message': 'Text cannot be None'}
    
    if not isinstance(text, str):
        return {'status': 'error', 'message': 'Text must be a string'}
    
    text = text.strip()
    
    if len(text) == 0:
        return {'status': 'error', 'message': 'Text cannot be empty'}
    
    if len(text) > 5000:
        return {'status': 'error', 'message': 'Text too long (max 5000 chars)'}
    
    try:
        # Vectorize and predict
        vec = vectorizer.transform([text])
        proba = model.predict_proba(vec)[0]
        
        # Get prediction based on threshold
        positive_prob = proba[1]
        prediction = 'positive' if positive_prob >= threshold else 'negative'
        confidence = max(proba)
        
        return {
            'status': 'success',
            'prediction': prediction,
            'confidence': round(float(confidence), 4),
            'probabilities': {
                'negative': round(float(proba[0]), 4),
                'positive': round(float(proba[1]), 4)
            }
        }
    except Exception as e:
        return {'status': 'error', 'message': str(e)}

# Test
for text in test_cases:
    res = serve_prediction_solution(text, result['model'], result['vectorizer'])
    print(f"Input: {str(text)[:20] if text else 'None'}... → Status: {res['status']}")

---
## 📋 Week 5 Practice Summary

**Exercises Completed:**
- [ ] Text Preprocessing Pipeline
- [ ] TF-IDF from Scratch
- [ ] Sentiment Classifier
- [ ] API Schema Design
- [ ] Production Prediction Function

**Key Concepts:**
- Text preprocessing steps
- Understanding TF-IDF internals
- Building text classifiers
- API design patterns
- Production-ready code

---
**Ready for Week 6: Debugging & Reasoning!** 🚀